# 02 - Exploratory Data Analysis

This notebook loads the raw DataFrame saved by `01_data_loading.ipynb` and produces four visualizations to build intuition about churn drivers before we preprocess and model the data.

## 1. Imports and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_pickle('../data/raw_df.pkl')
df.head()

## 2. Churn distribution

How many customers churned vs. stayed? This tells us how imbalanced the target class is, which motivates the use of SMOTE later in the preprocessing notebook.

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Churn', hue='Churn', palette='Set2', legend=False)
ax.set_title('Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Number of Customers')
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

**Insight:** The dataset is noticeably imbalanced — far more customers stay (`No`) than churn (`Yes`). A model trained naively on this data will be biased toward predicting the majority class, which is why we apply SMOTE to the training set during preprocessing.

## 3. Churn by contract type

Does the type of contract a customer is on relate to their likelihood of churning?

In [ ]:
plt.figure(figsize=(7, 4.5))
ax = sns.countplot(data=df, x='Contract', hue='Churn', palette='Set1')
ax.set_title('Churn by Contract Type')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.show()

**Insight:** Customers on **month-to-month** contracts churn at a much higher rate than those on one- or two-year contracts. Contract length looks like it will be one of the strongest predictors of churn.

## 4. Tenure vs. churn

A histogram of customer tenure (months with the company), split by churn status.

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True,
             palette='Set1', element='step', stat='density', common_norm=False)
plt.title('Tenure Distribution by Churn Status')
plt.xlabel('Tenure (months)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

**Insight:** Customers who churn tend to have **much shorter tenure** — churn is concentrated in the first few months. Customers who stay past roughly the two-year mark rarely churn, suggesting early onboarding/engagement is critical for retention.

## 5. Correlation heatmap of numeric features

We convert `TotalCharges` to numeric (coercing the blank strings to NaN, just for this exploratory view) and encode `Churn` as 0/1 so it can be included in the correlation matrix alongside the other numeric columns.

In [ ]:
numeric_df = df.copy()
numeric_df['TotalCharges'] = pd.to_numeric(numeric_df['TotalCharges'], errors='coerce')
numeric_df['Churn_encoded'] = numeric_df['Churn'].map({'Yes': 1, 'No': 0})

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_encoded']
corr = numeric_df[numeric_cols].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

**Insight:** `tenure` and `TotalCharges` are strongly positively correlated (longer-tenured customers have paid more in total, unsurprisingly). `Churn_encoded` correlates negatively with `tenure` and positively (weakly) with `MonthlyCharges`, consistent with the patterns seen in the previous charts: short-tenure, higher-bill customers churn more.

## Summary

- Churn is imbalanced (~26-27% positive class) → use SMOTE on train only.
- Month-to-month contracts churn far more than long-term contracts.
- Low tenure is strongly associated with churn.
- `tenure`, `Contract`, and `MonthlyCharges` look like key predictive features.

These observations carry into `03_preprocessing.ipynb`.